In [ ]:
import snowflake.connector
import requests
import json
import yaml
from datetime import datetime

In [ ]:
# Load config from git repo via Snowflake Git API Integration
# This assumes discovery-config.yml is available in the Snowflake stage or git repo
# For this notebook, we'll use inline config or read from a file

config = {
    "github": {
        "owner": "<GITHUB_OWNER>",
        "repo": "<GITHUB_REPO>",
        "branch": "main",
        "workflow_id": "discovery.yml"
    },
    "snowflake": {
        "databases": ["<DATABASE_NAME>"]
    },
    "github_token_secret": "<SNOWFLAKE_SECRET_NAME>"
}

# Alternatively, load from YAML config file if available
# try:
#     with open('discovery-config.yml', 'r') as f:
#         config = yaml.safe_load(f)
# except FileNotFoundError:
#     print("Config file not found, using inline config")

print("Config loaded successfully")

In [ ]:
# Query current Snowflake metadata (lightweight queries)
# This notebook uses lightweight queries to detect changes:
# - Table counts per schema
# - MAX(last_ddl) per schema
# - Column counts

# Get current session context for Snowflake connection
# In Snowflake Notebooks, connection is automatically available
try:
    # Use snowflake.connector with current session context
    # In Snowflake Notebooks, we can use the built-in connection
    ctx = snowflake.connector.connect(
        account=snowflake.connector.connect.__dict__.get('account'),
        user=snowflake.connector.connect.__dict__.get('user'),
        password=snowflake.connector.connect.__dict__.get('password')
    )
except Exception as e:
    # For testing, we'll use mock data
    print(f"Connection error: {e}")
    print("Using mock data for demonstration")
    ctx = None

current_state = {}

if ctx:
    cursor = ctx.cursor()
    
    try:
        for db in config["snowflake"]["databases"]:
            # Lightweight query: get table counts per schema
            cursor.execute(f"""
                SELECT TABLE_SCHEMA, COUNT(*) as table_count
                FROM {db}.INFORMATION_SCHEMA.TABLES
                WHERE TABLE_TYPE = 'BASE TABLE'
                GROUP BY TABLE_SCHEMA
            """)
            table_counts = cursor.fetchall()
            
            # Lightweight query: get MAX(last_ddl) per schema
            cursor.execute(f"""
                SELECT TABLE_SCHEMA, MAX(LAST_ALTERED) as max_last_ddl
                FROM {db}.INFORMATION_SCHEMA.TABLES
                WHERE TABLE_TYPE = 'BASE TABLE'
                GROUP BY TABLE_SCHEMA
            """)
            last_ddl = cursor.fetchall()
            
            # Lightweight query: get column counts per schema
            cursor.execute(f"""
                SELECT TABLE_SCHEMA, COUNT(*) as column_count
                FROM {db}.INFORMATION_SCHEMA.COLUMNS
                GROUP BY TABLE_SCHEMA
            """)
            column_counts = cursor.fetchall()
            
            # Store in current_state dict
            current_state[db] = {
                "table_counts": {schema: count for schema, count in table_counts},
                "last_ddl": {schema: str(ddl) for schema, ddl in last_ddl},
                "column_counts": {schema: count for schema, count in column_counts}
            }
            
    finally:
        cursor.close()
else:
    # Mock data for testing
    current_state = {
        "ANALYTICS": {
            "table_counts": {"PUBLIC": 5, "STAGING": 3},
            "last_ddl": {"PUBLIC": "2026-03-23 10:00:00", "STAGING": "2026-03-22 15:30:00"},
            "column_counts": {"PUBLIC": 25, "STAGING": 15}
        }
    }

print(f"Current state captured: {json.dumps(current_state, indent=2)}")

In [ ]:
# Fetch previous state from git repo via GitHub API
# Read discovery/_manifest.json from the git repository

previous_state = None

if ctx:
    # Get GitHub token from Snowflake secret
    try:
        cursor = ctx.cursor()
        cursor.execute(f"SELECT PARSE_JSON(system$get_secret('{config['github_token_secret']}'))")
        secret = cursor.fetchone()
        github_token = json.loads(secret[0])["github_token"]
        cursor.close()
    except Exception as e:
        print(f"Error fetching GitHub token: {e}")
        github_token = None
else:
    # For testing, use placeholder
    github_token = "<GITHUB_TOKEN>"

if github_token:
    owner = config["github"]["owner"]
    repo = config["github"]["repo"]
    branch = config["github"]["branch"]
    
    # GitHub API endpoint to fetch file content
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/discovery/_manifest.json?ref={branch}"
    headers = {
        "Authorization": f"token {github_token}",
        "Accept": "application/vnd.github.v3+json"
    }
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            content = response.json()
            # Decode base64 content
            import base64
            decoded_content = base64.b64decode(content["content"]).decode('utf-8')
            previous_state = json.loads(decoded_content)
            print(f"Previous state loaded successfully")
        elif response.status_code == 404:
            print("Manifest file not found (first run)")
            previous_state = None
        else:
            print(f"Error fetching manifest: {response.status_code}")
    except Exception as e:
        print(f"Error fetching previous state: {e}")
else:
    # Mock previous state for testing
    previous_state = {
        "ANALYTICS": {
            "table_counts": {"PUBLIC": 4, "STAGING": 3},
            "last_ddl": {"PUBLIC": "2026-03-22 10:00:00", "STAGING": "2026-03-22 15:30:00"},
            "column_counts": {"PUBLIC": 20, "STAGING": 15}
        }
    }

print(f"Previous state: {json.dumps(previous_state, indent=2) if previous_state else 'None'}")

In [ ]:
# Run diff engine to compare current and previous states
# Import from discovery.diff.engine module

# In a real implementation, this would import from the diff module:
# from discovery.diff.engine import DiffEngine, extract_current_state, load_previous_state
# diff_engine = DiffEngine()
# diff_result = diff_engine.compare(current_state, previous_state)

# For this notebook, we'll implement a simple diff
class DiffResult:
    def __init__(self, has_changes, added_objects, removed_objects, modified_objects, summary):
        self.has_changes = has_changes
        self.added_objects = added_objects
        self.removed_objects = removed_objects
        self.modified_objects = modified_objects
        self.summary = summary

def simple_diff(current, previous):
    """Simple diff implementation for lightweight metadata"""
    added = []
    removed = []
    modified = []
    
    if previous is None:
        # First run - all objects are "added"
        for db, db_data in current.items():
            for schema in db_data.get("table_counts", {}).keys():
                added.append(f"TABLE: {db}.{schema}.*")
        return DiffResult(
            has_changes=True,
            added_objects=added,
            removed_objects=[],
            modified_objects=[],
            summary=f"First run detected {len(added)} schemas"
        )
    
    # Compare table counts
    for db in current:
        if db not in previous:
            added.append(f"DATABASE: {db}")
            continue
        
        for schema in current[db].get("table_counts", {}):
            if schema not in previous[db].get("table_counts", {}):
                added.append(f"SCHEMA: {db}.{schema}")
            elif current[db]["table_counts"][schema] != previous[db]["table_counts"].get(schema):
                modified.append(f"TABLE_COUNT: {db}.{schema} ({previous[db]['table_counts'].get(schema, 0)} -> {current[db]['table_counts'][schema]})")
        
        for schema in previous[db].get("table_counts", {}):
            if schema not in current[db].get("table_counts", {}):
                removed.append(f"SCHEMA: {db}.{schema}")
    
    has_changes = len(added) > 0 or len(removed) > 0 or len(modified) > 0
    summary_parts = []
    if added:
        summary_parts.append(f"+{len(added)} added")
    if removed:
        summary_parts.append(f"-{len(removed)} removed")
    if modified:
        summary_parts.append(f"~{len(modified)} modified")
    summary = ", ".join(summary_parts) if summary_parts else "No changes"
    
    return DiffResult(
        has_changes=has_changes,
        added_objects=added,
        removed_objects=removed,
        modified_objects=modified,
        summary=summary
    )

diff_result = simple_diff(current_state, previous_state)

print(f"Diff Results:")
print(f"  has_changes: {diff_result.has_changes}")
print(f"  added_objects: {diff_result.added_objects}")
print(f"  removed_objects: {diff_result.removed_objects}")
print(f"  modified_objects: {diff_result.modified_objects}")
print(f"  summary: {diff_result.summary}")

In [ ]:
# If changes detected, trigger GitHub workflow_dispatch
# Use External Access Integration for GitHub API calls

if diff_result.has_changes:
    owner = config["github"]["owner"]
    repo = config["github"]["repo"]
    branch = config["github"]["branch"]
    workflow_id = config["github"]["workflow_id"]
    
    # GitHub API endpoint for workflow_dispatch
    url = f"https://api.github.com/repos/{owner}/{repo}/actions/workflows/{workflow_id}/dispatches"
    headers = {
        "Authorization": f"token {github_token}",
        "Accept": "application/vnd.github.v3+json"
    }
    
    payload = {
        "ref": branch,
        "inputs": {
            "diff_summary": diff_result.summary,
            "triggered_by": "snowflake-notebook"
        }
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 204:
            print(f"SUCCESS: Triggered discovery workflow: {diff_result.summary}")
        else:
            print(f"ERROR: Failed to trigger workflow: {response.status_code}")
            print(f"Response: {response.text}")
    except Exception as e:
        print(f"ERROR: Exception while triggering workflow: {e}")
else:
    print("No changes detected, skipping workflow trigger")

In [ ]:
# Log final result
log_entry = {
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "has_changes": diff_result.has_changes,
    "summary": diff_result.summary,
    "added_objects": diff_result.added_objects,
    "removed_objects": diff_result.removed_objects,
    "modified_objects": diff_result.modified_objects,
    "action_taken": "workflow_dispatch_triggered" if diff_result.has_changes else "no_action"
}

print("=" * 50)
print("DISCOVERY TRIGGER NOTEBOOK - FINAL LOG")
print("=" * 50)
print(f"Timestamp: {log_entry['timestamp']}")
print(f"Changes Detected: {log_entry['has_changes']}")
print(f"Summary: {log_entry['summary']}")
print(f"Action Taken: {log_entry['action_taken']}")
print("=" * 50)
print("Added Objects:")
for obj in diff_result.added_objects:
    print(f"  + {obj}")
print("Removed Objects:")
for obj in diff_result.removed_objects:
    print(f"  - {obj}")
print("Modified Objects:")
for obj in diff_result.modified_objects:
    print(f"  ~ {obj}")
print("=" * 50)

# Return log entry for downstream use
log_entry